# Lab 44: The Overfitting Trap

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-44.ipynb)

**What you will do:** compare a fixed-routine and a varied-routine study group on a novel-context test, pool the class's results, and reproduce the book's overfitting figure with a real scikit-learn model to see exactly where "more training accuracy" stops meaning "more real learning".

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 44, Group A studied 30 vocabulary words the same way for three days (same room, same
order, same time), while Group B studied the same words but varied the method, order, and
room each day. On Day 4, both groups were tested in a new environment with a scrambled order.
Group B typically outperforms Group A, even though Group A's sessions may have felt more
productive — the **desirable difficulties** effect (Bjork, 1994).

## Record your results

Enter the Day 4 recall counts for both groups (or, if you ran this on yourself over two
separate weeks, treat your fixed-routine week as "Group A" and your varied-routine week as
"Group B").

In [ ]:
# example data: replace with your own study results
total_words = 30
groupA_recalled = 18  # example data: fixed routine, tested in a new context/order on Day 4
groupB_recalled = 24  # example data: varied routine, tested in a new context/order on Day 4

results = pd.DataFrame({
    "group": ["A: fixed routine", "B: varied routine"],
    "recalled": [groupA_recalled, groupB_recalled],
    "total": [total_words, total_words],
})
results["proportion recalled"] = (results["recalled"] / results["total"]).round(2)
display(results)

results.set_index("group")["proportion recalled"].plot.bar(color=["grey", "steelblue"], figsize=(5, 3))
plt.ylim(0, 1); plt.ylabel("proportion recalled in novel test context"); plt.show()

## Compare

The book reports that Group B "will typically outperform" Group A, but gives no specific
percentage — the desirable-difficulties effect is described qualitatively here, not
quantified. We can still check the direction of your own comparison.

In [ ]:
gap = results.loc[1, "proportion recalled"] - results.loc[0, "proportion recalled"]
print(f"Group B minus Group A: {gap:+.0%}")
if gap > 0:
    print("In the direction Bjork's desirable-difficulties account predicts.")
else:
    print("Not in the expected direction here -- a single pair of study groups is a weak test; the")
    print("effect is usually shown across many participants and many novel-context tests, not one each.")
print("Note: the book gives no percentage improvement to match against here, only the qualitative")
print("claim that the varied routine 'typically' wins on a novel test.")

## The AI side

Figure 5.3 in the book shows overfitting in idealised form: a model that bends to fit every
data point, including the noise, versus one that captures the underlying pattern; and a
U-shaped curve where training error keeps falling as model complexity grows, while error on
new data falls only up to a point and then rises. Below, that figure is reproduced with a
real model. A set of noisy data points is fitted with polynomials of increasing degree —
higher degree means more "complexity", more free parameters to bend the curve with. We track
the error on the data the model was fitted on (training error) and on data it has never seen
(test error) at every degree.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(11)
n = 40
x = np.sort(rng.uniform(0, 1, n))
true_function = lambda t: np.sin(2 * np.pi * t)
y = true_function(x) + rng.normal(0, 0.25, n)  # the true pattern, plus noise
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=11)

degrees = list(range(1, 16))
train_errors, test_errors = [], []
for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(x_train.reshape(-1, 1), y_train)
    train_errors.append(mean_squared_error(y_train, model.predict(x_train.reshape(-1, 1))))
    test_errors.append(mean_squared_error(y_test, model.predict(x_test.reshape(-1, 1))))
best_degree = degrees[int(np.argmin(test_errors))]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

x_grid = np.linspace(0, 1, 200).reshape(-1, 1)
axes[0].scatter(x_train, y_train, color="grey", s=20, label="training data (with noise)")
axes[0].plot(x_grid, true_function(x_grid), color="black", linestyle=":", label="true pattern (unobserved)")
for d, colour, label in [(1, "steelblue", "degree 1 (underfit)"),
                          (best_degree, "green", f"degree {best_degree} (good fit)"),
                          (15, "firebrick", "degree 15 (overfit)")]:
    fit = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x_train.reshape(-1, 1), y_train)
    axes[0].plot(x_grid, fit.predict(x_grid), color=colour, label=label)
axes[0].set_ylim(-2, 2); axes[0].legend(fontsize=8); axes[0].set_title("(a) Three fits to the same noisy data")

axes[1].plot(degrees, train_errors, "o-", color="steelblue", label="training error")
axes[1].plot(degrees, test_errors, "o-", color="firebrick", label="test error")
axes[1].axvline(best_degree, color="grey", linestyle="--", label=f"lowest test error (degree {best_degree})")
axes[1].set_xlabel("polynomial degree (model complexity)"); axes[1].set_ylabel("mean squared error")
axes[1].legend(fontsize=8); axes[1].set_title("(b) Training error keeps falling; test error does not")
plt.tight_layout(); plt.show()

print(f"Lowest test error at degree {best_degree} (test MSE = {min(test_errors):.3f}).")
print(f"At degree 15, training MSE = {train_errors[-1]:.3f} but test MSE = {test_errors[-1]:.3f} --")
print("the model fits the noise in the training set almost perfectly, and pays for it on new data.")

ridge_model = make_pipeline(PolynomialFeatures(15), Ridge(alpha=0.01)).fit(x_train.reshape(-1, 1), y_train)
ridge_test_mse = mean_squared_error(y_test, ridge_model.predict(x_test.reshape(-1, 1)))
print(f"\nDegree-15 model without regularisation: test MSE = {test_errors[-1]:.3f}")
print(f"Degree-15 model WITH ridge regularisation:    test MSE = {ridge_test_mse:.3f}")
print("Ridge regularisation, like dropout or data augmentation, penalises the model for relying too")
print("heavily on any single feature -- forcing it to spread its 'trust' more evenly, the same way")
print("varying your study routine forces a word to be stored via more than one association.")

The two panels reproduce Figure 5.3 computationally: panel (a) shows the same qualitative
fits the caption describes (a curve that bends to the noise versus one that captures the
pattern); panel (b) shows the U-shaped complexity trade-off. Ridge regularisation recovers
much of the lost test accuracy at degree 15 without reducing the model's raw complexity — it
constrains how that complexity gets used. That is the machine-learning equivalent of
desirable difficulties: varying how you study does not reduce how much material you cover,
but it changes how the resulting memory is organised, making it less "overfitted" to one
specific room, order, or method.

## Pool the class data

Pool the class: each pair or group adds one row (an anonymous group ID, never names) with
their Group A and Group B recall counts on the Day 4 novel-context test. The example rows
below are placeholders so the cell runs.

In [ ]:
import io
csv_text = """group_id,total_words,groupA_recalled,groupB_recalled
T01,30,17,23
T02,30,20,22
T03,30,15,25
T04,30,19,19
T05,30,16,24
T06,30,21,20
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
class_df["A_prop"] = class_df.groupA_recalled / class_df.total_words
class_df["B_prop"] = class_df.groupB_recalled / class_df.total_words

boot_diff = []
for _ in range(5000):
    sample = class_df.sample(len(class_df), replace=True)  # resample whole rows to keep the pairing
    boot_diff.append((sample.B_prop - sample.A_prop).mean())
lo, hi = np.percentile(boot_diff, [2.5, 97.5])

print(f"Class mean recall -- Group A (fixed): {class_df.A_prop.mean():.0%}, "
      f"Group B (varied): {class_df.B_prop.mean():.0%}")
print(f"Mean B-minus-A advantage: {(class_df.B_prop - class_df.A_prop).mean():+.0%} "
      f"(95% bootstrap interval {lo:+.0%} to {hi:+.0%}, n = {len(class_df)} pairs)")

class_df[["A_prop", "B_prop"]].rename(
    columns={"A_prop": "Group A: fixed", "B_prop": "Group B: varied"}).plot.box(figsize=(5, 3))
plt.ylabel("proportion recalled, novel-context test"); plt.ylim(0, 1); plt.show()

## Questions to think about

1. In the ai_lab plot, at what degree does the test error start rising even though the training error keeps falling? Relate this turning point to the "novel test context" in Steps 3-4 of the experiment -- what would be the human-memory equivalent of testing at that same degree of complexity versus testing at a much higher one?
2. Ridge regularisation reduced the degree-15 model's test error without reducing its degree (its raw complexity). Bjork's desirable difficulties similarly do not reduce how much material Group B studies. What do the two interventions have in common, structurally?
3. Would you expect your own single Group A vs Group B comparison to be more or less reliable than the class's pooled result, and why?
4. The overfitting metaphor is also applied to political worldviews later in the chapter ("a narrative flexible enough to account for all the data"). Using the vocabulary from the ai_lab demo -- training error, test error, model complexity -- restate that claim about worldviews in your own words.

## Challenge

Pick a topic you already know reasonably well and deliberately "interleave" your review of it
with a second, unrelated topic — alternating between the two every few minutes rather than
blocking your study of each. Test yourself on both a week later and compare with how well you
retain material you study in one uninterrupted block.

In [ ]:
# Example data: replace with your own interleaved-vs-blocked comparison.
total_items = 20
blocked_recalled_1_week = 9       # example data: studied in one uninterrupted block
interleaved_recalled_1_week = 13  # example data: same amount of material, studied interleaved with another topic

print(f"Blocked study: {blocked_recalled_1_week}/{total_items} recalled after one week")
print(f"Interleaved study: {interleaved_recalled_1_week}/{total_items} recalled after one week")
print("Kornell and Bjork (2008) found that interleaving improves the ability to discriminate between")
print("categories, even though it feels harder during study -- the same 'desirable difficulty' logic")
print("as Group B's varied routine. The values above are example data -- replace them with your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-44.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")